<div dir="rtl" style="text-align:right">

# ارائه کامل پروژه SimSiam روی CIFAR-10

این notebook مسیر کامل پروژه را از معرفی ایده تا آموزش، ارزیابی و اجرای دمو نشان می‌دهد. متن‌های فارسی راست‌به‌چپ هستند و کدها برای اجرا در ریشه پروژه تنظیم شده‌اند.

</div>

<div dir="rtl" style="text-align:right">

## هدف پروژه

در مرحله اول encoder بدون استفاده از label، نمایش تصویری یاد می‌گیرد. سپس encoder ثابت می‌شود و یک classifier خطی برای ۱۰ کلاس CIFAR-10 آموزش می‌بیند. در پایان، نتیجه با supervised baseline مقایسه و مدل در Gradio آزمایش می‌شود.

</div>

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.seed import seed_everything
seed_everything(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

<div dir="rtl" style="text-align:right">

## ۱. تست معماری با batch ساختگی

این تست قبل از آموزش اجرا می‌شود تا مطمئن شویم ورودی با شکل `[batch, 3, 32, 32]` پذیرفته شده و loss اسکالر تولید می‌کند.

</div>

In [ ]:
from models.simsiam import SimSiam

simsiam = SimSiam().to(DEVICE)
x1 = torch.rand(4, 3, 32, 32, device=DEVICE)
x2 = torch.rand(4, 3, 32, 32, device=DEVICE)
loss = simsiam(x1, x2)
assert loss.ndim == 0
print('Input:', tuple(x1.shape))
print('Loss:', round(loss.item(), 4))
print('Architecture test: PASS')

<div dir="rtl" style="text-align:right">

## ۲. بارگذاری CIFAR-10

پوشه باید در `data/cifar-10-batches-py` باشد و شامل پنج batch آموزشی و یک batch آزمایشی باشد. برای تست سریع، `LIMIT` را کوچک می‌گذاریم؛ برای اجرای کامل مقدار آن را `None` کنید.

</div>

In [ ]:
from datasets.cifar10 import CIFAR10, CLASSES

LIMIT = 256
train_set = CIFAR10('data', train=True, download=False, two_views=True, limit=LIMIT)
test_set = CIFAR10('data', train=False, download=False, limit=LIMIT)
view1, view2, label = train_set[0]
assert tuple(view1.shape) == (3, 32, 32)
print('Train samples:', len(train_set))
print('Test samples:', len(test_set))
print('Classes:', ', '.join(CLASSES))
print('Sample:', tuple(view1.shape), 'label=', label)

<div dir="rtl" style="text-align:right">

## ۳. Self-supervised pretraining

دو view از هر تصویر وارد SimSiam می‌شوند. label در محاسبه loss استفاده نمی‌شود. اجرای این سلول checkpoint را در `checkpoints/notebook_simsiam.pt` ذخیره می‌کند.

برای اجرای کامل، `LIMIT = 50000` و تعداد epoch را افزایش دهید.

</div>

In [ ]:
from training.pretrain import train as pretrain

RUN_TRAINING = False  # برای اجرای مجدد به True تغییر دهید
if RUN_TRAINING:
    pretrained_model, pretrain_history = pretrain(
        epochs=5, batch_size=64, limit=50000, device=DEVICE,
        checkpoint='checkpoints/notebook_simsiam.pt'
    )
    print('Pretraining history:', pretrain_history)
else:
    pretrained_model, pretrain_history = None, []
    print('Pretraining skipped. Existing checkpoint can be used.')

<div dir="rtl" style="text-align:right">

## ۴. Linear evaluation

encoder از checkpoint بارگذاری و freeze می‌شود. فقط لایه خطی ۵۱۲ به ۱۰ آموزش می‌بیند. برای جلوگیری از اجرای ناخواسته طولانی، این سلول با `RUN_TRAINING` کنترل می‌شود.

</div>

In [ ]:
from training.linear_eval import train as linear_train

if RUN_TRAINING:
    classifier = linear_train(
        epochs=5, batch_size=64, limit=50000, device=DEVICE,
        encoder_checkpoint='checkpoints/simsiam.pt',
        checkpoint='checkpoints/linear.pt'
    )
    print('Linear training: PASS')
else:
    classifier = None
    print('Linear training skipped. Existing checkpoint can be evaluated.')

<div dir="rtl" style="text-align:right">

## ۵. ارزیابی واقعی روی test set

این سلول مدل‌های آموزش‌دیده را روی ۱۰٬۰۰۰ تصویر test ارزیابی می‌کند و accuracy هر روش را نمایش می‌دهد.

</div>

In [ ]:
from evaluation.evaluate import evaluate_linear, evaluate_supervised, save_report

linear_result = evaluate_linear(limit=10000, device=DEVICE)
supervised_result = evaluate_supervised(limit=10000, device=DEVICE)
report = {'linear': linear_result, 'supervised': supervised_result}
save_report(report, 'results/notebook_evaluation.json')
print(f"Linear accuracy: {linear_result['accuracy']:.2%}")
print(f"Supervised accuracy: {supervised_result['accuracy']:.2%}")
print('Report saved: results/notebook_evaluation.json')

<div dir="rtl" style="text-align:right">

## ۶. نمایش loss

اگر pretraining همین notebook اجرا شده باشد، روند loss نمایش داده می‌شود. اگر از checkpoint قبلی استفاده کنید، history در دسترس نیست و این موضوع طبیعی است.

</div>

In [ ]:
if pretrain_history:
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(pretrain_history) + 1), pretrain_history, marker='o')
    plt.title('SimSiam pretraining loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('Loss plot skipped: set RUN_TRAINING=True to train in this notebook.')

<div dir="rtl" style="text-align:right">

## ۷. تست checkpoint مفقود

پروژه در نبود checkpoint باید خطای واضح بدهد، نه اینکه بی‌صدا متوقف شود.

</div>

In [ ]:
from utils.checkpoint import load_checkpoint

try:
    load_checkpoint('checkpoints/file_that_does_not_exist.pt', SimSiam())
except FileNotFoundError as error:
    print('Expected error:', error)
    print('Missing-checkpoint handling: PASS')

<div dir="rtl" style="text-align:right">

## ۸. اجرای Gradio Demo


</div>

In [7]:

from app.demo import launch
launch('checkpoints/linear.pt')

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
from IPython.display import Image, display
display(Image(filename='results/accuracy_comparison.png'))

<div dir="rtl" style="text-align:right">

## جمع‌بندی برای ارائه

SimSiam بدون label ویژگی یاد می‌گیرد؛ stop-gradient از collapse جلوگیری می‌کند؛ linear evaluation کیفیت representation را می‌سنجد؛ و supervised baseline برای مقایسه استفاده می‌شود. دقت مدل فعلی به تعداد epoch و داده وابسته است و CIFAR-10 کلاس‌هایی مانند گوسفند یا پشم ندارد.

</div>